# Week 30, Day 4: Polymorphism and duck typing

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w030d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Duck:
    def sound(self):
        return "Quack"

class Robot:
    def sound(self):
        return "Beep"

class Kettle:
    def sound(self):
        return "Whee"

for thing in [Duck(), Robot(), Kettle()]:
    print(type(thing).__name__, thing.sound())
print(len("abc"), len([1, 2]), len({"a": 1}))

In [ ]:
class Car:
    def __init__(self, hours):
        self.hours = hours

class Van:
    def __init__(self, hours):
        self.hours = hours

class Bike:
    def __init__(self, hours):
        self.hours = hours

def hire_cost(vehicle):
    if isinstance(vehicle, Car):
        return 1500 * vehicle.hours
    elif isinstance(vehicle, Van):
        return 2500 * vehicle.hours + 1000
    elif isinstance(vehicle, Bike):
        return 300 * vehicle.hours
    raise TypeError("unknown vehicle")

print(sum(hire_cost(v) for v in [Car(2), Van(1), Bike(3)]))

In [ ]:
class Car:
    def __init__(self, hours):
        self.hours = hours

    def cost(self):
        return 1500 * self.hours

class Van:
    def __init__(self, hours):
        self.hours = hours

    def cost(self):
        return 2500 * self.hours + 1000

class Bike:
    def __init__(self, hours):
        self.hours = hours

    def cost(self):
        return 300 * self.hours

class Scooter:                       # added later: nothing else changes
    def __init__(self, hours):
        self.hours = hours

    def cost(self):
        return 200 * self.hours + 50

print(sum(v.cost() for v in [Car(2), Van(1), Bike(3), Scooter(1)]))

In [ ]:
def print_receipt(items, out):
    """items: pairs of (name, pence). out: anything with a write(text) method."""
    total = 0
    for name, pence in items:
        out.write(f"{name:<12}{pence / 100:>7.2f}\n")
        total += pence
    out.write(f"{'Total':<12}{total / 100:>7.2f}\n")

class Screen:
    def write(self, text):
        print(text, end="")

class Recorder:
    def __init__(self):
        self.lines = []

    def write(self, text):
        self.lines.append(text.rstrip("\n"))

basket = [("Suya", 450), ("Zobo", 180)]
print_receipt(basket, Screen())
r = Recorder()
print_receipt(basket, r)
print(r.lines)

In [ ]:
import io

def print_receipt(items, out):
    total = 0
    for name, pence in items:
        out.write(f"{name:<12}{pence / 100:>7.2f}\n")
        total += pence
    out.write(f"{'Total':<12}{total / 100:>7.2f}\n")

buffer = io.StringIO()
print_receipt([("Puff-puff", 300)], buffer)
print(repr(buffer.getvalue()))

with open("receipt.txt", "w") as f:
    print_receipt([("Puff-puff", 300)], f)
print(open("receipt.txt").read(), end="")

print("also", "works", file=buffer)
print(buffer.getvalue().splitlines()[-1])

In [ ]:
class Square:
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side ** 2

class Label:
    def __init__(self, text):
        self.text = text

things = [Square(3), Label("Kitchen"), Square(2)]

# look before you leap
print(sum(t.area() for t in things if hasattr(t, "area")))

# easier to ask forgiveness
total = 0
for t in things:
    try:
        area = t.area
    except AttributeError:
        continue
    total += area()
print(total)

In [ ]:
class Square:
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side ** 2

for t in [Square(3), "a label", 42]:
    area = getattr(t, "area", None)
    print(type(t).__name__, area() if area else "no area")

> **Key idea.** Polymorphism means one piece of code working with many classes, each responding its own way. Move `if isinstance` chains into a method on each class, so a new kind needs only a new class. Python is duck-typed: a function needs whatever methods it calls, not a particular class, and the built-ins (`for`, `len`, `+`, `sorted`, `print(file=...)`) work the same way. File-like objects such as `io.StringIO` can stand in for real files. Try and catch (EAFP) is the usual style, but keep the `try` around only the risky line; use `isinstance` for built-in types you can't add methods to and for checking outside input.

### Exercise: Fees without isinstance

Write three payment classes, each made with an `amount` in pence and each with a `fee()` method:

- `Cash`: no fee;
- `Card`: 1.5 per cent of the amount, rounded down (`amount * 15 // 1000`);
- `Transfer`: a flat 20p.

Then write `total_due(payments)`, the sum of every amount plus its fee. Don't use `isinstance` or `type`: let each payment work out its own fee.

In [ ]:
___

def total_due(payments):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fees():\n    \"fees\"\n    Ca, Cd, T = ns[\"Cash\"], ns[\"Card\"], ns[\"Transfer\"]\n    assert Ca(1000).fee() == 0 and Cd(1000).fee() == 15 and Cd(99).fee() == 1 and Cd(66).fee() == 0 and T(5).fee() == 20\n    assert total_due([Ca(500), Cd(2000), T(1000)]) == 3550 and total_due([]) == 0\ndef test_style():\n    \"no type checks\"\n    assert \"isinstance\" not in source and \"type(\" not in source, \"let each class work out its own fee\"")

### Exercise: Add anything

Write `combine(items)`, which adds all the items together with `+`, starting from the first one: `combine([1, 2, 3])` is 6, `combine(["ab", "c"])` is `"abc"`, `combine([[1], [2, 3]])` is `[1, 2, 3]`, and it works for any objects that support `+`. An empty list raises `ValueError`. The items passed in must not change.

In [ ]:
def combine(items):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"many types\"\n    assert combine([1, 2, 3]) == 6 and combine([2.5]) == 2.5 and combine([\"ab\", \"c\"]) == \"abc\"\n    assert combine([(1,), (2, 3)]) == (1, 2, 3)\n    first = [1]\n    got = combine([first, [2, 3]])\n    assert got == [1, 2, 3] and first == [1], \"the first list was changed\"\n    class Pence:\n        def __init__(self, p):\n            self.p = p\n        def __add__(self, other):\n            return Pence(self.p + other.p)\n    assert combine([Pence(5), Pence(7), Pence(1)]).p == 13\n    try:\n        combine([])\n        assert False, \"an empty list should raise ValueError\"\n    except ValueError:\n        pass")

### Exercise: A file that counts

Write `save_lines(lines, out)`, which writes each string in `lines` to `out` followed by a newline, using only `out.write`, and returns how many lines it wrote. Then write `WriteCounter`, a class that can be passed as `out`: it doesn't store the text, it only counts, keeping `chars` (characters written) and `newlines` (newline characters written). `save_lines` must also work with a real file and with `io.StringIO`.

In [ ]:
def save_lines(lines, out):
    ___

class WriteCounter:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_counter():\n    \"counting\"\n    W = ns[\"WriteCounter\"]\n    w = W()\n    assert (w.chars, w.newlines) == (0, 0)\n    assert save_lines([\"Lagos\", \"\", \"Leeds\"], w) == 3\n    assert (w.chars, w.newlines) == (13, 3), f\"got {w.chars}, {w.newlines}\"\n    w.write(\"a\\nb\\n\")\n    assert (w.chars, w.newlines) == (17, 5)\ndef test_files():\n    \"real files\"\n    import io\n    buf = io.StringIO()\n    assert save_lines([\"one\", \"two\"], buf) == 2 and buf.getvalue() == \"one\\ntwo\\n\"\n    with open(\"out.txt\", \"w\") as f:\n        save_lines([\"x\"], f)\n    assert read_file(\"out.txt\") == \"x\\n\" and save_lines([], io.StringIO()) == 0")

### Exercise: Skip, but don't hide

Write `total_area(things)`: the sum of `thing.area()` for every thing that has an `area` method, skipping things that don't. An `AttributeError` raised **inside** an `area` method is a real bug and must not be hidden: it should come out of `total_area`.

In [ ]:
def total_area(things):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"areas\"\n    class Sq:\n        def __init__(self, s):\n            self.s = s\n        def area(self):\n            return self.s ** 2\n    assert total_area([Sq(3), \"label\", 7, Sq(2), None]) == 13 and total_area([]) == 0\ndef test_bug_not_hidden():\n    \"bugs inside area() aren't hidden\"\n    class Broken:\n        def area(self):\n            return self.sidee ** 2\n    try:\n        total_area([Broken()])\n        assert False, \"the AttributeError inside Broken.area should come out\"\n    except AttributeError:\n        pass")

### Bug hunt: Only lists allowed

`average` refuses a tuple of marks and a range, though it could handle both. Fix it so this prints `60.0` twice.

In [ ]:
def average(values):
    if type(values) != list:
        raise TypeError("average needs a list")
    if len(values) == 0:
        raise ValueError("no values")
    return sum(values) / len(values)

print(average((50, 70, 60)))
print(average(range(40, 81, 10)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"both averages\"\n    expect_output(\"60.0\\n60.0\")\n    assert average([1, 2]) == 1.5\n    try:\n        average(())\n        assert False, \"an empty tuple should still raise ValueError\"\n    except ValueError:\n        pass")

### Bug hunt: The silent typo

The basket should total `450`, but it prints `150`, with no error at all. The broad `try` is hiding a bug. Find the real mistake and fix it.

In [ ]:
class Item:
    def __init__(self, name, pence):
        self.name = name
        self.pence = pence

    def price(self):
        return self.pence

class Offer(Item):
    def price(self):
        return self.pense // 2

def basket_total(things):
    total = 0
    for thing in things:
        try:
            total += thing.price()
        except AttributeError:
            pass
    return total

print(basket_total([Item("Rice", 150), Offer("Oil", 600), "voucher"]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the total\"\n    expect_output(\"450\")")

### Bug hunt: A tram that doesn't fit

The journey planner crashes with `AttributeError: 'Tram' object has no attribute 'fare'`. Make `Tram` fit the same interface as the others, so it prints `735`.

In [ ]:
class Bus:
    def fare(self, stops):
        return 175 if stops <= 10 else 250

class Train:
    def fare(self, stops):
        return 120 * stops

class Tram:
    def price(self, stops):
        return 100 + 20 * stops

legs = [(Bus(), 4), (Train(), 3), (Tram(), 5)]
print(sum(mode.fare(stops) for mode, stops in legs))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the journey\"\n    expect_output(\"735\")")

### Puzzle: Any kind of writer

Arrange the lines into a function that writes a heading and a rule to any object with a `write` method, and a class that collects what's written. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
class A:
    def size(self):
        return 1

class B:
    def size(self):
        return 2

class C:
    pass

things = [A(), B(), C(), A()]

p = ___   # sum(t.size() for t in things if hasattr(t, "size"))
q = ___   # [type(t).__name__ for t in things if not hasattr(t, "size")]
r = ___   # getattr(C(), "size", 99)
s = ___   # sorted(["b", "A", "a"])
t = ___   # sum([[1], [2]], [])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 4, \"p\"\n    assert q == [\"C\"], \"q\"\n    assert (r, s, t) == (99, [\"A\", \"a\", \"b\"], [1, 2])")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a text canvas

Draw shapes on a grid of characters. Every shape has a `char` attribute (the character it's drawn with) and a `cells()` method returning a **set** of `(row, col)` pairs that it covers.

- `Dot(row, col, char="*")`: one cell.
- `HLine(row, col, length, char="-")`: `length` cells going right from `(row, col)`. `length` below 1 raises `ValueError`.
- `Box(top, left, height, width, char="#")`: the **outline** of a rectangle `height` rows by `width` columns, with its top-left corner at `(top, left)`. `height` or `width` below 1 raises `ValueError`.
- `render(shapes, height, width)` returns a string of `height` lines, each `width` characters, joined by `"\n"`. Empty cells are `.`. Shapes are drawn in order, so later ones cover earlier ones. Cells outside the canvas are ignored.

`render` must work with **any** object that has `char` and `cells()`, not just these three classes.

In [ ]:
___

def render(shapes, height, width):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_shapes():\n    \"the shapes' cells\"\n    D, H, B = ns[\"Dot\"], ns[\"HLine\"], ns[\"Box\"]\n    assert D(1, 2).cells() == {(1, 2)} and D(0, 0).char == \"*\"\n    assert H(2, 1, 3).cells() == {(2, 1), (2, 2), (2, 3)} and H(0, 0, 1, \"=\").char == \"=\"\n    assert B(0, 0, 3, 3).cells() == {(0, 0), (0, 1), (0, 2), (1, 0), (1, 2), (2, 0), (2, 1), (2, 2)}\n    assert B(1, 1, 1, 2).cells() == {(1, 1), (1, 2)} and len(B(0, 0, 4, 5).cells()) == 14\n    for make in [lambda: H(0, 0, 0), lambda: B(0, 0, 0, 3), lambda: B(0, 0, 2, -1)]:\n        try:\n            make()\n            assert False, \"sizes below 1 should raise ValueError\"\n        except ValueError:\n            pass\ndef test_render():\n    \"drawing\"\n    D, H, B = ns[\"Dot\"], ns[\"HLine\"], ns[\"Box\"]\n    got = render([B(0, 0, 3, 5), H(1, 1, 3, \"~\"), D(1, 2, \"o\")], 4, 6)\n    assert got == \"#####.\\n#~o~#.\\n#####.\\n......\", f\"got:\\n{got}\"\n    assert render([], 2, 3) == \"...\\n...\"\n    assert render([H(0, 3, 10), D(-1, 0), D(5, 5)], 2, 5) == \"...--\\n.....\"\ndef test_duck():\n    \"any object with char and cells()\"\n    class Diagonal:\n        char = \"\\\\\"\n        def cells(self):\n            return {(0, 0), (1, 1), (2, 2)}\n    assert render([Diagonal()], 3, 3) == \"\\\\..\\n.\\\\.\\n..\\\\\"")

## Warm-up

The three classes share nothing (no parent but `object`), yet one loop calls `sound()` on all of them and each answers in its own way. Python never asked what they were; it only asked whether each one had a `sound` method when the call happened. `len` works the same way on a string, a list and a dictionary. Today is about making use of that.

## One call, many behaviours

**Polymorphism** (Greek for "many shapes") means one piece of code working with objects of different classes, each responding in its own way. You've used it all week: `shape.area()` gave a rectangle's or a circle's area depending on the object.

Its biggest practical use is getting rid of code like this:

Every new kind of vehicle means finding this function (and every other one like it) and adding a branch. Move each branch into its own class instead, as a method with the same name:

- The knowledge about each vehicle lives with that vehicle.
- Code that **uses** vehicles (`sum(v.cost() ...)`) never changes when a new kind arrives.
- The classes could share a parent (it would be a good place for the `__init__` they all repeat), but the loop doesn't need one.

## Duck typing

"If it walks like a duck and quacks like a duck, it's a duck." Python treats objects that way: a function that calls `x.write(text)` works for **any** `x` with a `write` method that accepts a string. Nothing checks the class. This is **duck typing**, and it means a function's real requirement is the set of methods it uses, not a class name.

So when you write a function, the question is "what does it need its arguments to do?":

`print_receipt` doesn't know about `Screen` or `Recorder`, and they don't share a parent. It works for both, and for a real file, as the next section shows.

The cost of duck typing is that mistakes show up late. Pass something with no `write` and you get an `AttributeError` from inside the function, at the moment it first calls `out.write`. Clear docstrings ("anything with a `write(text)` method") are how duck-typed code tells callers what it needs. Week 33's type hints, and Week 32's Protocols, let tools check it before the program runs.

## Python's own ducks

Python's built-ins are duck-typed through and through:

| You write | Python needs the object to... |
|---|---|
| `for x in obj` | be iterable |
| `len(obj)` | have a length (a `__len__` method) |
| `a + b` | support `+` (an `__add__`, as in Week 29) |
| `sorted(items)` | support `<` between the items |
| `print(x)` | have a `__str__` (every object inherits one) |
| `print(..., file=f)`, `json.dump(data, f)` | have a `write` method |

So `sum` adds anything that supports `+`, `sorted` sorts anything that supports `<`, and `print` writes to anything with `write`. Week 31 tours the special methods behind each row.

**File-like objects** are the most useful duck of all. Anything with `write` can take the place of a file being written, and anything with `read` or iteration over lines can stand in for one being read. The standard library's `io.StringIO` is a file that lives in memory, which is very handy for tests:

Same function, three destinations: memory, a file on disk, and (through `print`) an object `print` itself writes into.

## Ask first or try?

When an object might not have what you need, there are two styles:

- **Look before you leap** (LBYL): check first, with `hasattr(obj, "name")` or `isinstance`.
- **Easier to ask forgiveness than permission** (EAFP): just try it, and catch the error if it fails.

Python code leans towards EAFP, because checking first can be wrong (the check and the use can disagree) and doubles the work. But a `try` must be **narrow**. Look at where the `try` ends in the example: only the lookup of `t.area` is inside it, and the **call** `area()` is outside. If the call were inside, an `AttributeError` caused by a bug **inside** some `area` method (a misspelt attribute, say) would be caught too, and that shape would be silently skipped. You'll fix exactly that in the practice.

`getattr(obj, "name", default)` is a neat middle way: it returns the attribute, or the default if there isn't one, with no `try` at all:

## When isinstance is right

Duck typing doesn't make `isinstance` wrong. It's the right tool when:

- **the types behave differently and you can't add methods to them**, such as accepting either a single string or a list of strings (`if isinstance(names, str): names = [names]`). You can't give `str` a new method.
- **checking input from outside your program** (a file, a form, JSON), where a clear `TypeError` or `ValueError` early is better than a confusing failure later.
- **catching exceptions**, which is `isinstance` under another name.

What's usually wrong is checking for a **specific class** when the code only needs a **behaviour**: `if type(values) != list` turns away tuples, ranges and your own classes, all of which would have worked.

## Practice

## Quiz

## Daily challenge

## Before you go

Search your own code from earlier weeks for an `if`/`elif` chain that checks what kind of thing something is, and ask whether each branch belongs in a method instead. Tomorrow: composition versus inheritance, and why "has a" is often a better design than "is a".